In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path
# Load variables from .env into environment
load_dotenv()

# Read the key
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
MODEL_NAME = os.getenv("MODEL_NAME")
VISION_MODEL_NAME = os.getenv("VISION_MODEL_NAME")

In [2]:

current_dir = Path.cwd()
BASE_DIR = current_dir.parent.parent
root_folder = BASE_DIR / "dataset/"
# List only directories inside subfolder
sub_folders = [f for f in root_folder.iterdir() if f.is_dir()]

doc_dict = {}

for folder in sub_folders:
    parent_folder = BASE_DIR / folder.name
    subfolder = parent_folder.joinpath(folder)
    iterator = list(subfolder.iterdir())
    first_two = iterator[0:1]   # TypeError!
    ext = os.path.splitext(first_two[0])[1][1:]
    if ext.lower() in ['jpg', 'jpeg', 'png', 'tiff', 'bmp', 'gif']: 
        # doc_dict[folder.name] = extract_text_from_image(str(first_two[0]))
        doc_dict[folder.name] = str(first_two[0])

In [3]:
import base64
import os
import mimetypes
import json
from pathlib import Path
from typing import Union, Optional, Dict, Any, List
from groq import Groq

# Initialize once — reuses the connection
_client: Optional[Groq] = None

def _get_client() -> Groq:
    global _client
    if _client is None:
        api_key = os.environ.get("GROQ_API_KEY")
        if not api_key:
            raise ValueError("GROQ_API_KEY environment variable not set.")
        _client = Groq(api_key=api_key)
    return _client


def _encode_image(image_path: str) -> tuple[str, str]:
    """Returns (base64_string, mime_type)."""
    path = Path(image_path)
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {image_path}")

    mime_type, _ = mimetypes.guess_type(path.name)
    mime_type = mime_type or "image/jpeg"

    with open(path, "rb") as f:
        encoded = base64.b64encode(f.read()).decode("utf-8")
    return encoded, mime_type


def read_image(image: Union[str, List[str]],prompt: str = "Extract all text and key data from this image.", model: str = "qwen/qwen3.8-27b",*,
    json_schema: Optional[Dict[str, Any]] = None,temperature: float = 1.0,max_tokens: int = 2048,stream: bool = False,) -> Union[str, Dict[str, Any]]:
    """
    Read and extract data from one or more images using Groq's vision models.

    Args:
        image:       Local file path, URL, or a list of either (max 3).
        prompt:      Instruction for the model.
        model:       Groq vision model ID.
        json_schema: Optional JSON schema — if provided, returns a dict.
        temperature: Sampling temperature.
        max_tokens:  Max completion tokens.
        stream:      Stream the response (returns generator).

    Returns:
        String, dict (if json_schema given), or generator (if stream=True).
    """
    client = _get_client()

    # Normalize to list
    images = [image] if isinstance(image, str) else image
    if not images:
        raise ValueError("At least one image is required.")
    if len(images) > 3:
        raise ValueError("Groq supports a maximum of 3 images per request.")

    # Build content blocks
    content: List[Dict[str, Any]] = [{"type": "text", "text": prompt}]

    for img in images:
        if img.startswith(("http://", "https://")):
            url = img
        else:
            b64, mime = _encode_image(img)
            url = f"data:{mime};base64,{b64}"
        content.append({
            "type": "image_url",
            "image_url": {"url": url},
        })

    # Build request kwargs
    kwargs: Dict[str, Any] = {
        "model": model,
        "messages": [{"role": "user", "content": content}],
        "temperature": temperature,
        "max_completion_tokens": max_tokens,
        "stream": stream,
    }

    if json_schema:
        kwargs["response_format"] = {"type": "json_object"}
        # Inject schema into the prompt so the model knows what to produce
        schema_hint = (
            "\n\nRespond ONLY with valid JSON matching this schema:\n"
            f"{json.dumps(json_schema, indent=2)}"
        )
        content[0]["text"] += schema_hint

    completion = client.chat.completions.create(**kwargs)

    if stream:
        return _stream_response(completion)

    raw = completion.choices[0].message.content

    if json_schema:
        try:
            return json.loads(raw)
        except json.JSONDecodeError as e:
            raise ValueError(f"Model did not return valid JSON: {raw}") from e

    return raw


def _stream_response(completion):
    """Yield text chunks from a streaming completion."""
    for chunk in completion:
        delta = chunk.choices[0].delta.content
        if delta:
            yield delta

In [4]:
kyc_raw = read_image(doc_dict['id_documents'])
print(kyc_raw)

Based on the image provided, here is the extracted text and key data:

**Header Information:**
*   **Company Name:** MediShield Health Insurance Ltd.
*   **Card Type:** GOLD PLAN MEMBER ID CARD
*   **Plan/Group Reference:** MED-GLD | Group: GRP-41029

**Member Details:**
*   **Member Name:** John Johnson L.
*   **Member ID:** MED-GLD-3592974
*   **Group Number:** GRP-41029
*   **Effective Date:** 07/01/2025
*   **Expiration Date:** (Field is blank, but see status below)

**Card Status:**
*   **Alert:** "** CARD EXPIRED — NOT VALID **" (Displayed in a red-bordered box)

**Cost Sharing Summary:**
*   **PCP Visit:** $25 copay
*   **Specialist:** $60 copay
*   **Emergency Room:** $250 copay
*   **Urgent Care:** $50 copay
*   **Deductible (Ind.):** $1,500
*   **Out-of-Pocket Max:** $7,500
*   **Rx Tier 1 (Generic):** $15 / 30-day

**Other Elements:**
*   A barcode is visible at the bottom left.
*   A square, pixelated logo (resembling a stylized 'E' or grid) is visible in the bottom right c

In [5]:
import os
import json
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain.agents import create_agent
import re

load_dotenv()

@tool
def check_insurance_validity(text: str) -> str:
    """If the text appears to be an insurance document, check whether it
    contains a 'NOT VALID' or 'CARD EXPIRED' marking.
    Returns JSON: {"is_insurance": bool, "not_valid_found": bool, "status": "..."}"""

    # 1. Detect if this looks like an insurance document
    insurance_keywords = [
        r"INSURANCE",
        r"INSURED",
        r"POLICY\s*(?:NUMBER|NO|#)",
        r"MEMBER\s*(?:ID|NUMBER|NO|#)",
        r"CLAIM\s*FORM",
        r"HEALTH\s*PLAN",
        r"COVERAGE",
    ]

    is_insurance = any(
        re.search(kw, text, re.IGNORECASE) for kw in insurance_keywords
    )

    if not is_insurance:
        return json.dumps({
            "is_insurance": False,
            "not_valid_found": False,
            "status": "NOT_AN_INSURANCE_DOCUMENT",
        })

    # 2. Look for a 'NOT VALID' marking
    #    Handles: NOT VALID | NOT-VALID | NOTVALID | "NOT VALID" | INVALID
    not_valid_pattern = r"\bNOT[\s\-]?VALID\b|\bINVALID\b"
    m = re.search(not_valid_pattern, text, re.IGNORECASE)

    if m:
        return json.dumps({
            "is_insurance": True,
            "not_valid_found": True,
            "matched_text": m.group(0),
            "status": "expired_id",
        })

    return json.dumps({
        "is_insurance": True,
        "not_valid_found": False,
        "status": "valid",
    })


# ---------- AGENT ----------
def build_agent():
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0,
        api_key=os.getenv("GROQ_API_KEY"),
    )
    return create_agent(
        model=llm,
        tools=[check_insurance_validity],
        system_prompt=(
            "You extract information from CLAIM forms. When given text, call the appropriate tool once "
            "and reply with ONLY the extracted information as plain text. If not found, reply NOT_FOUND."
        ),
    )


# ---------- PUBLIC API ----------
def insurance_validity(text: str) -> str:
    agent = build_agent()
    result = agent.invoke({"messages": [{"role": "user", "content": text}]})
    return result["messages"][-1].content.strip()




In [6]:
raws = insurance_validity(kyc_raw)
raws

'CARD EXPIRED — NOT VALID'

In [7]:
import os
import re
import json
from langchain_groq import ChatGroq
from langchain.agents import create_agent
from langchain_core.tools import tool


@tool
def check_insurance_validity(text: str) -> str:
    """If the text appears to be an insurance document, check whether it
    contains a 'NOT VALID' marking or a 'CARD EXPIRED' marking.
    Returns JSON: {"is_insurance": bool, "not_valid_found": bool, "status": "..."}"""

    # 1. Detect if this looks like an insurance document
    insurance_keywords = [
        r"INSURANCE",
        r"INSURED",
        r"POLICY\s*(?:NUMBER|NO|#)",
        r"MEMBER\s*(?:ID|NUMBER|NO|#)",
        r"CLAIM\s*FORM",
        r"HEALTH\s*PLAN",
        r"COVERAGE",
    ]

    is_insurance = any(
        re.search(kw, text, re.IGNORECASE) for kw in insurance_keywords
    )

    if not is_insurance:
        return json.dumps({
            "is_insurance": False,
            "not_valid_found": False,
            "status": "NOT_AN_INSURANCE_DOCUMENT",
        })

    # 2. Look for invalid markings:
    #    - NOT VALID | NOT-VALID | NOTVALID | INVALID
    #    - CARD EXPIRED | CARD-EXPIRED | EXPIRED CARD
    invalid_patterns = [
        (r"\bNOT[\s\-]?VALID\b",        "NOT_VALID"),
        (r"\bINVALID\b",                "INVALID"),
        (r"\bCARD[\s\-]?EXPIRED\b",     "CARD_EXPIRED"),
        (r"\bEXPIRED[\s\-]?CARD\b",     "EXPIRED_CARD"),
    ]

    for pattern, label in invalid_patterns:
        m = re.search(pattern, text, re.IGNORECASE)
        if m:
            return json.dumps({
                "is_insurance": True,
                "not_valid_found": True,
                "matched_text": m.group(0),
                "matched_type": label,
                "status": "expired_id",
            })

    return json.dumps({
        "is_insurance": True,
        "not_valid_found": False,
        "status": "valid",
    })


# ---------- AGENT ----------
def build_agent():
    llm = ChatGroq(
        model="openai/gpt-oss-120b",
        temperature=0,
        api_key=os.getenv("GROQ_API_KEY"),
    )
    return create_agent(
        model=llm,
        tools=[check_insurance_validity],
        system_prompt=(
            "You extract information from CLAIM forms. When given text, call the appropriate tool once "
            "and reply with ONLY the extracted information as plain text. If not found, reply NOT_FOUND."
        ),
    )


# ---------- PUBLIC API ----------
def insurance_validity(text: str) -> dict:
    """
    Returns:
        {
            "kyc_passed": bool,
            "flags": [list of strings],
            "confidence": float
        }

    Invalid scenarios:
        - "NOT VALID" / "NOT-VALID" / "NOTVALID" / "INVALID"
        - "CARD EXPIRED" / "CARD-EXPIRED" / "EXPIRED CARD"
    """
    agent = build_agent()
    result = agent.invoke({"messages": [{"role": "user", "content": text}]})
    raw = result["messages"][-1].content.strip()

    # Try to parse the agent's response as JSON; fall back to running the tool directly
    try:
        data = json.loads(raw)
    except (json.JSONDecodeError, TypeError):
        data = json.loads(check_insurance_validity.invoke({"text": text}))

    flags = []
    kyc_passed = False
    confidence = 0.0

    if not data.get("is_insurance", False):
        flags.append("NOT_AN_INSURANCE_DOCUMENT")
        kyc_passed = False
        confidence = 0.95

    elif data.get("not_valid_found", False):
        matched_type = data.get("matched_type", "UNKNOWN")
        flags.append(matched_type)                                 # e.g. "NOT_VALID" or "CARD_EXPIRED"
        flags.append(f"MATCHED: {data.get('matched_text', 'UNKNOWN')}")
        kyc_passed = False
        confidence = 0.98

    else:
        kyc_passed = True
        confidence = 0.90

    return {
        "kyc_passed": kyc_passed,
        "flags": flags,
        "confidence": confidence,
    }

In [8]:
insurance_validity(kyc_raw)

{'kyc_passed': False,
 'flags': ['NOT_VALID', 'MATCHED: NOT VALID'],
 'confidence': 0.98}